# Full A9 pipeline (Vast.ai, single GPU): tokenizer retrain -> pretrain -> SFT -> full eval

A9 = `--vocab-size=16384 --depth=7` (default aspect_ratio=64 -> model_dim=512), sized against
`d6`'s ~73.53M param budget (72.35M actual, non-embedding fraction 30.4% vs `d6`'s 14.4% --
see docs/RESEARCH_LOG.md). Isolates "reallocate the embedding-dominated param budget toward a
smaller vocab" as a lever, independent of A10's "more depth at fixed width".

**Two things this notebook is careful about, both learned the hard way on A10 (see
docs/RESEARCH_LOG.md 2026-08-11):**

1. **Tokenizer collision.** `nanochat/tokenizer.py` always writes to `{NANOCHAT_BASE_DIR}/tokenizer`
   -- there's no vocab-size-specific path. Retraining in place would silently overwrite the
   `vocab_size=32768` tokenizer that `d4`/`d6`/`a10` all depend on, locally *and* on Drive once
   synced. This notebook uses a **separate `NANOCHAT_BASE_DIR` for A9** entirely
   (`~/nanochat_cache_a9`), with the pretraining data shards **symlinked** in from the shared
   cache (no re-download) and the tokenizer trained fresh into this isolated dir, synced to a
   **distinct Drive path** (`gdrive:tokenizer_a9`, not `gdrive:tokenizer`).
2. **Disk-full checkpoint accumulation.** `save_checkpoint()` has no retention policy -- every
   `--save-every` interval stays on disk forever, which crashed A10's pretrain once already.
   This notebook runs `vastai/prune_checkpoints.py` in the background alongside
   `kaggle/sync_checkpoints.py`, keeping only the last 3 local steps once each has had a chance
   to sync to Drive -- bounded disk usage regardless of how many total steps the run saves.

Run this in the browser Jupyter app on a Vast.ai instance (reuses `~/repo` and
`~/.config/rclone/rclone.conf` if this is the same box A10 ran on). Download when done
(File -> Download) and archive under `vastai/runs/`, same convention as `kaggle/runs/`.

No credentials are stored in this file -- Cell 2 reuses an existing `rclone.conf` if present,
only prompting via `getpass` as a fallback. Never hardcode real credentials here, even
temporarily -- this repo is public and git history keeps anything committed, even after it's
later removed.

## Cell 0: clean up build caches + check what's using disk

In [ ]:
import subprocess

print("Before cleanup:")
!df -h /

!rm -rf ~/.cache/uv ~/.cache/pip
!rm -rf ~/.cargo/registry/cache ~/.cargo/registry/src
subprocess.run(["bash", "-lc", "apt-get clean 2>/dev/null || true"])

print("After cleanup:")
!df -h /
print()
print("Largest local dirs (for reference, in case more cleanup is needed):")
!du -sh ~/nanochat_cache/* 2>/dev/null | sort -rh
!du -sh ~/nanochat_cache_a9/* 2>/dev/null | sort -rh

## Cell 1: repo + deps (reuses `~/repo` if already set up)

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = os.path.expanduser("~/repo")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml
!uv pip install --system --python {sys.executable} accelerate  # for vram_probe.py

# Every later shell cell uses PY (not bare `python3`) -- a bare `python3` subprocess can
# resolve to a different interpreter than sys.executable and miss what was just installed
# above (hit this exact bug on the A10 eval notebook).
PY = sys.executable
print(f"Cell 1 done. PY={PY}")

## Cell 2: rclone config + separate A9 base dir, symlink shared dataset (no re-download)

In [ ]:
import os

rclone_conf_path = os.path.expanduser("~/.config/rclone/rclone.conf")
if os.path.exists(rclone_conf_path):
    print(f"Found existing {rclone_conf_path} -- reusing it, no prompt needed.")
else:
    print("No rclone.conf on this box yet. Enter the same 4 credentials used before:")
    from getpass import getpass
    client_id = getpass("GDRIVE_CLIENT_ID: ")
    client_secret = getpass("GDRIVE_CLIENT_SECRET: ")
    oauth_token = getpass("GDRIVE_OAUTH_TOKEN (the whole JSON blob): ")
    folder_id = getpass("GDRIVE_FOLDER_ID: ")

    os.makedirs(os.path.dirname(rclone_conf_path), exist_ok=True)
    with open(rclone_conf_path, "w") as f:
        f.write(
            "[gdrive]\n"
            "type = drive\n"
            "scope = drive\n"
            f"client_id = {client_id}\n"
            f"client_secret = {client_secret}\n"
            f"token = {oauth_token}\n"
            f"root_folder_id = {folder_id}\n"
            "team_drive =\n"
        )
    del client_id, client_secret, oauth_token, folder_id

# Separate base dir for A9 -- keeps its tokenizer (vocab_size=16384) from ever colliding with
# the shared vocab_size=32768 tokenizer d4/d6/a10 use.
NANOCHAT_BASE_DIR = os.path.expanduser("~/nanochat_cache_a9")
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)

# Reuse the pretraining data shards from the shared cache via symlink -- same ClimbMix data
# works for any vocab_size, no reason to re-download 45 shards. Falls back to a fresh Drive
# pull if this is a clean instance with no shared cache yet.
shared_data_dir = os.path.expanduser("~/nanochat_cache/base_data_climbmix")
a9_data_dir = os.path.join(NANOCHAT_BASE_DIR, "base_data_climbmix")
if os.path.isdir(shared_data_dir) and not os.path.exists(a9_data_dir):
    os.symlink(shared_data_dir, a9_data_dir)
    print(f"Symlinked {a9_data_dir} -> {shared_data_dir} (no re-download)")
elif not os.path.isdir(a9_data_dir):
    print("No shared dataset cache found, pulling from Drive...")
    !rclone copy gdrive:base_data_climbmix {a9_data_dir} --checksum -v

!ls {a9_data_dir} | head -5
!df -h /

## Cell 3: train the A9 tokenizer (vocab_size=16384, isolated dir + Drive path)

In [ ]:
import os
os.chdir(os.path.expanduser("~/repo"))

a9_tokenizer_pkl = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "tokenizer", "tokenizer.pkl")
if os.path.exists(a9_tokenizer_pkl):
    print(f"A9 tokenizer already present at {a9_tokenizer_pkl}, skipping.")
else:
    !{PY} -m scripts.tok_train --vocab-size=16384 --max-chars=2000000000
    # Distinct remote path -- never gdrive:tokenizer, that's the shared 32768 one.
    !rclone copy {os.path.dirname(a9_tokenizer_pkl)} gdrive:tokenizer_a9 --checksum -v

!ls {os.path.dirname(a9_tokenizer_pkl)}

## Cell 4: VRAM probe -- largest safe --device-batch-size for this config on this GPU

In [ ]:
import re

# total_batch_size=262144 (auto) must divide evenly by device_batch_size * max_seq_len(2048) *
# world_size(1) -- so device_batch_size must divide 262144/2048 = 128 (same constraint A10 hit).
DIVISOR_TARGET = 128
probe_out = subprocess.run(
    [PY, "kaggle/vram_probe.py", "--depth=7", "--vocab-size=16384", "--max-seq-len=2048",
     f"--starting-batch-size={DIVISOR_TARGET}"],
    capture_output=True, text=True, cwd=os.path.expanduser("~/repo"),
)
print(probe_out.stdout)
print(probe_out.stderr)

m = re.search(r"Largest working --device-batch-size.*: (\d+)", probe_out.stdout)
probe_batch = int(m.group(1)) if m else 8
print(f"Probe found: {probe_batch}")

DEVICE_BATCH_SIZE = 8
for candidate in (128, 64, 32, 16, 8):
    if candidate <= DIVISOR_TARGET and candidate <= probe_batch:
        DEVICE_BATCH_SIZE = candidate
        break
print(f"Using --device-batch-size={DEVICE_BATCH_SIZE} (probe ceiling {probe_batch}, rounded down to a clean divisor of {DIVISOR_TARGET})")

## Cell 5: pretrain (background sync + checkpoint pruning alongside)

In [ ]:
import subprocess
import time

# --skip-subdirs tokenizer: this box's NANOCHAT_BASE_DIR holds A9's vocab_size=16384 tokenizer,
# but sync_checkpoints.py's "tokenizer" subdir always maps to the shared gdrive:tokenizer remote
# (vocab_size=32768, used by d4/d6/a10) -- without this flag the background sync would silently
# overwrite that shared tokenizer every poll. Cell 3 already pushed A9's tokenizer to the
# correct, separate gdrive:tokenizer_a9 path once; it doesn't change again during training.
sync_proc = subprocess.Popen(
    [PY, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--interval", "60",
     "--skip-subdirs", "tokenizer", "--log-file", os.path.expanduser("~/sync.log")],
    cwd=os.path.expanduser("~/repo"),
    env={**os.environ},
)
prune_proc = subprocess.Popen(
    [PY, "vastai/prune_checkpoints.py", "--model-tag", "a9", "--checkpoint-type", "base",
     "--keep", "3", "--min-age", "90", "--interval", "60", "--log-file", os.path.expanduser("~/prune.log")],
    cwd=os.path.expanduser("~/repo"),
    env={**os.environ},
)
time.sleep(2)
print(f"sync PID {sync_proc.pid}, prune PID {prune_proc.pid} running in background.")

!{PY} -m scripts.base_train \
    --depth=7 --vocab-size=16384 --window-pattern=L \
    --device-batch-size={DEVICE_BATCH_SIZE} --target-param-data-ratio=20 \
    --save-every=100 --run=dummy --model-tag=a9

sync_proc.terminate()
prune_proc.terminate()
!{PY} kaggle/sync_checkpoints.py --remote gdrive: --once --skip-subdirs tokenizer --log-file ~/sync.log
!df -h /

## Cell 6: SFT (background sync alongside -- SFT only saves once at the end, pruning not needed here)

In [ ]:
import subprocess

sync_proc = subprocess.Popen(
    [PY, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--interval", "60",
     "--skip-subdirs", "tokenizer", "--log-file", os.path.expanduser("~/sync.log")],
    cwd=os.path.expanduser("~/repo"),
    env={**os.environ},
)

!{PY} -m scripts.chat_sft \
    --model-tag=a9 --mmlu-epochs=0 --gsm8k-epochs=0 \
    --num-iterations=500 --chatcore-every=-1 --eval-every=100 --run=dummy

sync_proc.terminate()
!{PY} kaggle/sync_checkpoints.py --remote gdrive: --once --skip-subdirs tokenizer --log-file ~/sync.log
!df -h /

## Cell 7: free the pretrain checkpoint (already synced, SFT no longer needs it) + quick chat test

In [ ]:
import shutil

base_ckpt_dir = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "base_checkpoints", "a9")
if os.path.isdir(base_ckpt_dir):
    print(f"Removing {base_ckpt_dir} (already on Drive, not needed past this point)...")
    shutil.rmtree(base_ckpt_dir)
!df -h /

!{PY} -m scripts.chat_cli -i sft -g a9 -p "hi"
!{PY} -m scripts.chat_cli -i sft -g a9 -p "What is your name?"
!{PY} -m scripts.eval_repetition -i sft -g a9 --repetition-penalty 1.2 --no-repeat-ngram-size 3

## Cell 8: full chat_eval.py -- a9

In [ ]:
!{PY} -m scripts.chat_eval -i sft -g a9 2>&1 | tee ~/chat_eval_a9.log

## Cell 9: full BLiMP eval -- a9

In [ ]:
!{PY} -m scripts.eval_blimp -i sft -g a9 --batch-size 64 2>&1 | tee ~/blimp_a9.log